# Section 1: Problem and Population (text cell) One paragraph. Persona, failure point, SDG. Note anything that changed since M1 and why.

Bay Area poverty and food insecurity affect many low-income families, especially single mothers in Oakland who work part-time but still cannot afford consistent groceries because of high rent, transportation costs, and unstable income. My persona is Maria, a single mother in Oakland who needs food assistance for herself and her children. The main failure point is that resources such as food banks, CalFresh, and community aid programs exist, but the information is scattered, hard to understand, and does not clearly tell her what action to take.
Since Milestone 1, the project has become more focused on food insecurity instead of poverty in general because food assistance is a clearer problem for AI to support through text extraction and image analysis.

# Section 2: Proposed System (text cell) Input → AI processing → output → real-world action. Diagram or structured description.

Input:
Maria enters a short message about her situation, such as "I am a single mother in Oaland and need free groceries this week"

AI Processing:
The AI takes the important details from the text such as location, type of need, urgency, and possible eligibility

Output:
The system gives a structured summary of Maria's need, urgency level, reccomended service type, and a plain-language next step

Real-world action:
Maria can use the output to contact a food bank, apply for Calfresh, visit a food distribution site, or call a city/community service


# Section 3:  Project Code (code + text cells) Select at least two labs that best support your system and modify to your project. For each: include the code cell, preserve the output, and explain in a text cell what it demonstrates about your system. The selection and explanation are the work. You are encouraged to use AI to help you debug your code.

Lab 2 Part 3:

In [ ]:
import google.generativeai as genai
import json
import time
from google.colab import userdata

# Configure the API key using Colab's secrets manager
# Ensure you have stored your API key in Colab secrets under the name 'GOOGLE_API_KEY'
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

schema_prompt = """
You are an AI assistant that extracts structured information from resident messages about requests for assistance.
Your task is to identify and extract the following details into a JSON object:
- 'location': The city or specific location mentioned in the message.
- 'waste_type': The type of assistance or need the resident has (e.g., 'food assistance', 'housing', 'utilities', 'transportation').
- 'urgency': The level of urgency of the request ('LOW', 'MEDIUM', 'HIGH', 'IMMEDIATE').
- 'department': The most appropriate city department or service to handle this request (e.g., 'Social Services', 'Food Bank', 'Housing Authority').
- 'resident_language': The primary language the resident appears to be using in their message.
"""

resident_message = "I am a single mother in Oakland and need free groceries this week."

def extract_structured(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=schema_prompt
    )

    response = m.generate_content(
        message,
        generation_config={
            "temperature": 0.0,
            "response_mime_type": "application/json",
            "response_schema": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "waste_type": {"type": "string"},
                    "urgency": {"type": "string"},
                    "department": {"type": "string"},
                    "resident_language": {"type": "string"}
                },
                "required": [
                    "location",
                    "waste_type",
                    "urgency",
                    "department",
                    "resident_language"
                ]
            }
        }
    )

    time.sleep(12) # Added a sleep to prevent rate limiting issues with frequent calls
    return json.loads(response.text)

print("--- Resident message ---")
print(resident_message)
print("\n--- Structured extraction (run 3 times — format is identical each time) ---")

for i in range(1, 4):
    print(f"\nRun {i}:")
    result = extract_structured(resident_message)
    print(json.dumps(result, indent=2, ensure_ascii=False))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

Explanation/Analysis: Lab 2 supports my system because it shows how AI can convert a resident’s unstructured message into a consistent structured format. In this code, the function extract_structured(message) uses Gemini with a fixed JSON schema to extract key fields: location, waste_type, urgency, department, and resident_language. For my poverty-focused project, these same fields can be adapted to food insecurity by changing waste_type to need_type or assistance_type, such as food, housing, utilities, or transportation.

Lab 3 Part 3:

In [ ]:
import google.generativeai as genai
from IPython.display import Image, display # For displaying images, if needed
from PIL import Image # Uncomment if you need to load local images with PIL
from google.colab import userdata

# NOTE: You need to define image_filename and potentially configure genai.
# For example, if you have an image file named 'my_image.jpg' in the current directory:
image_filename = 'my_image.jpg' # Replace with your actual image path

# Configure genai with your API key from Colab secrets
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

civic_questions = [
    ("PROBLEM",  "Describe the civic or environmental problem visible in this image. Be specific about what you see."),
    ("IMPACT",   "What are the public health, safety, or community impacts of what is shown in this image?"),
    ("URGENCY",  "Rate urgency as LOW, MEDIUM, or HIGH. Start your answer with the label, then give a 1–2 sentence explanation."),
    ("ACTION",   "Which specific city department or service should respond to this? What action should they take?")
]

civic_system_instruction = """
You are analyzing a civic issue for a city 311 system.

Rules:
- Be clear, specific, and concise
- Do not include extra commentary
- Follow instructions exactly for each question
- For URGENCY, you must choose ONLY one: LOW, MEDIUM, or HIGH
"""

civic_results = {"answers": {}, "total_tokens": 0}

# Placeholder for analyze_image function
def analyze_image(image_path, prompt):
    # This is a placeholder. You'll need to implement actual image analysis logic.
    # For example, using a Gemini Vision model:
    try:
        img = Image.open(image_path) # Requires 'PIL' library and actual image file
        model = genai.GenerativeModel('gemini-pro-vision')
        response = model.generate_content([prompt, img])
        # Assuming response structure has a 'text' attribute for answer
        answer = response.text
        # Placeholder for usage, replace with actual token count if available
        usage = type('Usage', (object,), {'total_token_count': 50})() # Mock object
        return answer, usage
    except Exception as e:
        return f"Error analyzing image: {e}", type('Usage', (object,), {'total_token_count': 0})()

for label, question in civic_questions:
    print(f"--- {label} ---")

    full_prompt = civic_system_instruction + "\n\n" + question
    answer, usage = analyze_image(image_filename, full_prompt)

    civic_results["answers"][label] = answer
    civic_results["total_tokens"] += usage.total_token_count

    print(answer)
    print()

print(f"--- Total tokens used: {civic_results['total_tokens']} ---")
print("Running on Gemini free tier — no cost.")

SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

Explanation/Analysis: Lab 3 helps show how AI can look at an image and break it down into four simple questions what the problem is, why it matters, how urgent it is, and who should respond. In my project, this could help someone like Maria take a picture of a food pantry, community center, or even a flyer and quickly understand if it can help her and what she should do next. But when I ran the code, it actually gave an error saying the image file wasn’t found, which shows a real limitation since the system only works if the image is uploaded correctly. That matters because people dealing with poverty might not always have the time, tech skills, or reliable devices to do that. So while this lab shows how useful image analysis could be, it also highlights that the system needs to be simple, flexible, and not depend only on images to work properly.

# Section 4 — Edge Case Elicitation (code + text cells) Design and run a prompt intended to surface a failure in your system. Target one of: an input the system wasn't designed for, a user outside the assumed majority, or a condition that makes the AI's confidence misleading. Preserve the output. In a text cell, document the prompt, the output, and a one-sentence assessment: failure, near-miss, or acceptable.

In [ ]:
edge_case_prompt = """
A user writes: I just lost my job, I have no money, and my kids have not eaten today.
I do not speak English well and I do not have transportation. What should I do?
"""

result = extract_structured(edge_case_prompt)
print(result)


NameError: name 'extract_structured' is not defined

Explanation/Analysis:
This edge case tests a user who falls outside the typical assumption because they have multiple barriers at once, including urgent food insecurity, limited English, and no transportation. The system is able to identify the general need, such as food assistance and urgency, but it may not fully account for language barriers or the lack of transportation when suggesting next steps. This means the AI might recommend resources that are difficult for the user to access or understand.

Assessment:
Near miss, the system correctly identifies the problem and urgency, but does not fully adapt its recommendations to the user’s constraints, which could limit its real-world usefulness.